---

### 🎓 **Professor**: Apostolos Filippas

### 📘 **Class**: Web Analytics

### 📋 **Topic**: Data visualization with Seaborn (self-study)


🚫 **Note**: You are not allowed to share the contents of this notebook with anyone outside this class without written permission by the professor.

---

> **Before you begin:** [Update your course files and Python environment safely.](https://github.com/apostolosfilippas/wa/blob/main/assets/before-you-begin.md)



# 1. Data Visualization with Seaborn

The purpose of this notebook is to teach you some basic elements of visualization in Python. We've already used seaborn in class. Now let's see a few more things it can do!

Run the cells in order, using the course's `.venv` kernel. If you haven't updated the environment yet, run `uv sync` in your terminal in the `wa` folder. Both datasets are already in our `data` folder, so there is nothing else to download.

### Data Source: Survey on Mental Health in the Tech Workplace in 2014

- Our data comes from a 2014 survey about mental health and the workplace. A copy is in `data/mental_survey.csv`.
- The original course file is [here](https://drive.google.com/file/d/1JFvtWmu9qV68dZpJEPEethYiNGh_U-uE/view). More survey data can be found on [OSMI](https://osmihelp.org/research).
- This is an old, self-selected survey, not a representative sample of all tech workers. We are using it to learn how to plot data, not to estimate how common mental health conditions are today.
- Here are the features of the dataset:

| Feature | Description |
| --- | --- |
| Timestamp |The time when the survey is conducted  |
|Age|Age|
|Gender|Gender, entered as free text by the respondent|
|Country| Country|
|state|If you live in the United States, which state or territory do you live in?|
|self_employed|Are you self-employed?|
|family_history| Do you have a family history of mental illness?|
|treatment|Have you sought treatment for a mental health condition?|
|work_interfere|If you have a mental health condition, do you feel that it interferes with your work?|
|no_employees | How many employees does your company or organization have?|
|remote_work | Do you work remotely (outside of an office) at least 50% of the time?|
|tech_company | Is your employer primarily a tech company/organization?|
| benefits | Does your employer provide mental health benefits?|
| care_options | Do you know the options for mental health care your employer provides?|
| wellness_program | Has your employer ever discussed mental health as part of an employee wellness program?|
| seek_help | Does your employer provide resources to learn more about mental health issues and how to seek help?|
| anonymity | Is your anonymity protected if you choose to take advantage of mental health or substance abuse treatment resources?|
| leave | How easy is it for you to take medical leave for a mental health condition?|
| mental_health_consequence | Do you think that discussing a mental health issue with your employer would have negative consequences?|
|phys_health_consequence | Do you think that discussing a physical health issue with your employer would have negative consequences?|
| coworkers | Would you be willing to discuss a mental health issue with your coworkers?|
| supervisor | Would you be willing to discuss a mental health issue with your direct supervisor(s)?|
| mental_health_interview | Would you bring up a mental health issue with a potential employer in an interview?|
|phys_health_interview | Would you bring up a physical health issue with a potential employer in an interview?|
|mental_vs_physical|Do you feel that your employer takes mental health as seriously as physical health?|
|obs_consequence|Have you heard of or observed negative consequences for coworkers with mental health conditions in your workplace?|
|comments|Any additional notes or comments|

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from matplotlib.dates import DateFormatter
from pathlib import Path

%matplotlib inline

# the same basic style we've been using in class
plot_color = '#4477AA'
yes_no_palette = {'No': '#4477AA', 'Yes': '#EE7733'}
sns.set_theme(style='ticks', palette=list(yes_no_palette.values()),
              font_scale=1.1, rc={'figure.figsize': (10, 6)})

# notebooks can start in wa, scripts, or homework; let's find the course folder
course_folder = Path.cwd().resolve()
if course_folder.name in {'scripts', 'homework'}:
    course_folder = course_folder.parent
if not (course_folder / 'pyproject.toml').is_file():
    raise RuntimeError('Open the wa course folder in VS Code, then restart this notebook.')

data_folder = course_folder / 'data'
print('Course folder:', course_folder)

In [ ]:
# read our local copy, just as we did in the previous lectures
survey_df = pd.read_csv(data_folder / 'mental_survey.csv')
df = survey_df.copy()  # keep the original data around while we explore
df.head()

In [ ]:
# a quick view of missing values and distinct non-missing answers
pd.concat([df.isna().sum().rename('num_missing'),
           df.nunique().rename('num_unique')], axis=1)

## Example: Bar chart for missing values

Before making plots, let's see which columns have missing values. A bar chart makes it easy to compare how much is missing in each column.

In [ ]:
# isna() gives us True for a missing value; its mean gives us the missing fraction
missing_values = df.isna().mean()
missing_values = missing_values[missing_values > 0].sort_values(ascending=False)
missing_values

In [ ]:
missing_table = missing_values.rename_axis('feature').reset_index(name='fraction')
fig, ax = plt.subplots()
sns.barplot(data=missing_table, y='feature', x='fraction',
            color=plot_color, errorbar=None, ax=ax)
ax.xaxis.set_major_formatter(PercentFormatter(xmax=1))
ax.set(title='Missing values in the survey', ylabel='Feature',
       xlabel='Percentage of responses missing', xlim=(0, 1))
plt.tight_layout()
plt.show()

## Example: barplot

This example avoids `countplot` on purpose: we'll count the observations ourselves, and then use `barplot`. Notice how seaborn works with matplotlib's `fig` and `ax` if we want to customize the details.

A quick reminder:

- **Counter** counts how many times each value appears.
- **List comprehensions** let us create a list in one line. We saw these earlier! The example below keeps the even numbers from 0 to 9.

In [ ]:
# quick example: list comprehension
[y for y in range(10) if y % 2 == 0]

In [ ]:
from collections import Counter

country_count = Counter(df['Country'].dropna().tolist()).most_common(10)
country_idx = [country[0] for country in country_count]
country_val = [country[1] for country in country_count]

In [ ]:
fig, ax = plt.subplots()
sns.barplot(y=country_idx, x=country_val, color=plot_color, errorbar=None, ax=ax)
ax.set(title='Top ten countries in the survey', ylabel='Country',
       xlabel='Number of respondents')
plt.tight_layout()
plt.show()

## Example: catplot

`catplot` lets us compare categories. With `kind='count'`, the height of each bar is the number of observations. `hue` gives us different colors, and `col` gives us separate panels.

* self_employed - Are you self-employed?
* remote_work - Do you work remotely (outside of an office) at least 50% of the time?
* tech_company - Is your employer primarily a tech company/organization?

In [ ]:
g = sns.catplot(data=df, x='self_employed', hue='remote_work',
                col='tech_company', kind='count', order=['No', 'Yes'],
                hue_order=['No', 'Yes'], col_order=['No', 'Yes'],
                palette=yes_no_palette, height=4, aspect=1.1)
g.set_axis_labels('Self-employed?', 'Number of respondents')
g.set_titles('Tech company: {col_name}')
sns.move_legend(g, 'center left', bbox_to_anchor=(1, 0.5), title='Works remotely?')
g.figure.suptitle('Work arrangements in the survey', y=1.05)
plt.show()

## Example: Histogram

A histogram puts numeric values into bins and counts how many observations fall in each bin. `displot` creates the figure for us; `histplot` can draw a histogram on an existing axis. We'll use both below.

In [ ]:
# use describe() to display basic stats
df['Age'].describe()

Some ages are clearly impossible. For these examples, let's keep adults aged 18 to 100. This is a simple teaching rule, not proof that every remaining answer is correct!

We'll print how many rows we remove, rather than quietly changing the data.

In [ ]:
rows_before = len(df)
df = df.loc[df['Age'].between(18, 100)].copy()
print('Rows removed by the age filter:', rows_before - len(df))
print('Rows remaining:', len(df))

g = sns.displot(data=df, x='Age', bins=24, color=plot_color, height=4, aspect=1.7)
g.set_axis_labels('Age (years)', 'Number of respondents')
g.ax.set_title('Ages of respondents kept for our examples')
plt.show()

## Example: Box Plot

A box plot gives us a compact view of a numeric variable. The box shows the middle 50% of the observations, and the line inside it shows the median.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
sns.boxplot(data=df, x='Age', color=plot_color, ax=ax)
ax.set(title='Distribution of respondent ages', xlabel='Age (years)')
plt.tight_layout()
plt.show()

How to read this horizontal box plot:

- The left edge of the box is the first quartile, Q1: 25% of observations are at or below it.
- The line inside the box is the median: half the observations are at or below it. It doesn't have to sit in the center of the box!
- The right edge is the third quartile, Q3: 75% of observations are at or below it.
- The interquartile range, or IQR, is Q3 minus Q1.
- By default, the whiskers reach the most extreme observations within 1.5 IQR below Q1 and above Q3. They are NOT necessarily the minimum and maximum!
- Dots beyond the whiskers are possible outliers. A dot is not automatically a mistake.

See [seaborn's boxplot examples](https://seaborn.pydata.org/generated/seaborn.boxplot.html) if you want to play with the options.

## Example: FacetGrid

A `FacetGrid` gives us separate panels for different groups. Let's compare the age distributions of people who reported seeking treatment and people who did not.

We'll use the same bins in both panels. `stat='density'` makes the area under each histogram equal to 1, so we can compare shapes even when the groups have different sizes. The smooth curve is an estimate of the distribution, not extra data.

* treatment: Have you sought treatment for a mental health condition?

In [ ]:
# use the SAME bin edges in both panels
age_bins = list(range(18, 105, 5))
g = sns.FacetGrid(df, col='treatment', col_order=['No', 'Yes'], height=4, aspect=1.2)
g.map_dataframe(sns.histplot, x='Age', bins=age_bins, stat='density',
                kde=True, color=plot_color)
g.set_axis_labels('Age (years)', 'Density')
g.set_titles('Sought treatment: {col_name}')
g.figure.suptitle('Age distributions by reported treatment-seeking', y=1.05)
plt.show()

## Example: Grouped bars with catplot

Now let's compare the share who reported seeking treatment across age groups and work arrangements.

We'll turn Yes into 1 and No into 0 in a NEW column. Why? The mean of a 0/1 column is the share of 1s. If three out of five people answered Yes, the mean is 0.6, or 60%.

This tells us about **reported treatment-seeking**, not whether someone has a mental health condition.

In [ ]:
# put ages into groups, keeping the original Age column unchanged
age_labels = ['18-20', '21-30', '31-65', '66-100']
df['age_range'] = pd.cut(df['Age'], [17, 20, 30, 65, 100], labels=age_labels)

# keep the original text answers as well
df['treatment_yes'] = df['treatment'].map({'Yes': 1, 'No': 0})

A quick detour: the Gender column was free text, so it has lots of different answers. We can tidy capitalization and extra spaces, but that's NOT the same as deciding what someone's identity is.

We'll keep the original answers, and won't combine nonbinary, agender, queer, and other responses into one invented category. For the grouped charts below, we'll use remote-work status instead.

In [ ]:
# a small text-cleaning example, without overwriting the original answers
df['gender_clean'] = df['Gender'].str.strip().str.casefold()
df[['Gender', 'gender_clean']].drop_duplicates().head(12)

In [ ]:
# always check the group sizes before reading too much into the bars
group_sizes = pd.crosstab(df['age_range'], df['remote_work'])
print('Respondents in each age / remote-work group:')
print(group_sizes)

g = sns.catplot(data=df, x='age_range', y='treatment_yes', hue='remote_work',
                kind='bar', errorbar=None, order=age_labels,
                hue_order=['No', 'Yes'], palette=yes_no_palette,
                height=4, aspect=1.7)
g.set_axis_labels('Age group (years)', 'Share reporting treatment-seeking')
g.set(ylim=(0, 1))
g.ax.yaxis.set_major_formatter(PercentFormatter(xmax=1))
g.ax.set_title('Reported treatment-seeking by age and remote work')
sns.move_legend(g, 'center left', bbox_to_anchor=(1, 0.5), title='Works remotely?')
plt.show()

Some groups are tiny! A bar at 100% could be just ONE person. These bars describe our survey respondents; don't turn them into claims about everyone in that age group.

Similarly, we can compare reported treatment-seeking by family history and remote-work status. Here we'll add 95% bootstrap confidence intervals using `errorbar=('ci', 95)`. They show sampling uncertainty under the method's assumptions; they don't fix a self-selected sample or show that one thing causes another. `seed=42` keeps the intervals the same when we rerun the cell.

In [ ]:
print('Respondents in each family-history / remote-work group:')
print(pd.crosstab(df['family_history'], df['remote_work']))

g = sns.catplot(data=df, x='family_history', y='treatment_yes', hue='remote_work',
                kind='bar', errorbar=('ci', 95), seed=42,
                order=['No', 'Yes'], hue_order=['No', 'Yes'],
                palette=yes_no_palette, height=4, aspect=1.7)
g.set_axis_labels('Family history of mental illness?',
                  'Share reporting treatment-seeking')
g.set(ylim=(0, 1))
g.ax.yaxis.set_major_formatter(PercentFormatter(xmax=1))
g.ax.set_title('Reported treatment-seeking by family history and remote work')
sns.move_legend(g, 'center left', bbox_to_anchor=(1, 0.5), title='Works remotely?')
plt.show()

## Example: Time series plot

Seaborn can also plot multiple time series with a line chart. Let's switch datasets and look at US Treasury yields in October 2022.

A copy of the [Treasury's October 2022 table](https://home.treasury.gov/resource-center/data-chart-center/interest-rates/TextView?type=daily_treasury_yield_curve&field_tdr_date_value_month=202210) is in `data/treasury_yields_2022_10.csv`. We read it from the repo, just like the survey. No web request or extra HTML parser needed!

`1 Mo`, `2 Mo`, `3 Mo`, and `6 Mo` are maturities: one, two, three, and six months. The numbers are annualized yields in percent: 3.26 means 3.26%, not 326%. This is a fixed historical example, not today's rates.

In [ ]:
treasury_df = pd.read_csv(data_folder / 'treasury_yields_2022_10.csv',
                          parse_dates=['Date'])
yield_columns = ['1 Mo', '2 Mo', '3 Mo', '6 Mo']

# put the dates on the index, and keep them in chronological order
df_line = treasury_df.set_index('Date')[yield_columns].sort_index()
df_line.head()

In [ ]:
fig, ax = plt.subplots()
sns.lineplot(data=df_line, dashes=False,
             palette=['#4477AA', '#EE7733', '#228833', '#AA3377'], ax=ax)
ax.set(title='US Treasury yields, October 2022', xlabel='Date',
       ylabel='Annualized yield (%)')
ax.set_xlim(df_line.index.min(), df_line.index.max())
ax.set_xticks(pd.date_range(df_line.index.min(), df_line.index.max(), freq='7D'))
ax.xaxis.set_major_formatter(DateFormatter('%b %d'))
ax.legend(title='Maturity')
plt.tight_layout()
plt.show()

In [ ]:
# one dot per date; regplot adds a fitted straight line
# ci=None hides the usual confidence band; daily rates are not independent observations
fig, ax = plt.subplots()
sns.regplot(data=df_line, x='1 Mo', y='2 Mo', color=plot_color, ci=None, ax=ax)
ax.set(title='One-month and two-month yields, October 2022',
       xlabel='One-month annualized yield (%)',
       ylabel='Two-month annualized yield (%)')
plt.tight_layout()
plt.show()

## Example: Correlation Matrix

When exploring data, it's natural to check which variables move together. Let's use the four numeric yield columns we just plotted.

Don't assign arbitrary numbers to categories and then correlate those numbers! Coding No, Maybe, and Yes as 0, 1, and 2 would invent a spacing between the answers. That's not what we want to teach here.

In [ ]:
# use the same dates for every pair; keep the survey and its ages unchanged
corr_df = df_line[yield_columns].copy()
rows_before = len(corr_df)
corr_df = corr_df.dropna(subset=yield_columns)
print('Dates removed because a yield was missing:', rows_before - len(corr_df))
print('Dates used:', len(corr_df))

In [ ]:
# Pearson correlation, using the actual numeric yields
corrmat = corr_df.corr()
corrmat

A correlation ranges from -1 to 1. Values near 1 mean the variables tend to rise and fall together; values near -1 mean they tend to move in opposite directions. Near zero means little LINEAR relationship, not necessarily no relationship at all.

Correlation doesn't tell us that one variable causes the other. Here, rates also share a time trend, and we're looking at just one month. The fitted line above and the heatmap below are descriptions, not an investment model or an independent-observation experiment.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(corrmat, annot=True, fmt='.2f', square=True,
            vmin=-1, vmax=1, center=0, cmap='vlag',
            cbar_kws={'label': 'Pearson correlation'}, ax=ax)
ax.set_title('Treasury yield correlations, October 2022')
ax.tick_params(axis='both', labelrotation=0)
plt.tight_layout()
plt.show()

Notice the diagonal: every variable has a correlation of 1 with itself. The other squares compare different maturities. Use the numbers AND the colors when reading the chart.

The labels are the same maturities as before:

- `1 Mo`: one-month yield.
- `2 Mo`: two-month yield.
- `3 Mo`: three-month yield.
- `6 Mo`: six-month yield.

We haven't replaced the yields with category codes or changed the survey's ages. Those are still the original numbers.

## Which yields move most closely with the six-month yield?

In [ ]:
# remove the six-month yield itself, otherwise its correlation of 1 wins!
yield_correlations = corrmat['6 Mo'].drop('6 Mo')
# rank by strength, but keep the signed correlation values for the plot
ranked = yield_correlations.abs().sort_values(ascending=False).index
correlation_table = yield_correlations.loc[ranked].rename_axis('maturity').reset_index(name='correlation')

fig, ax = plt.subplots(figsize=(9, 4))
sns.barplot(data=correlation_table, x='correlation', y='maturity',
            color=plot_color, errorbar=None, ax=ax)
ax.axvline(0, color='0.3', linewidth=1)
ax.set(xlim=(-1, 1), title='Correlation with the six-month yield, October 2022',
       xlabel='Pearson correlation', ylabel='Maturity')
plt.tight_layout()
plt.show()

This ranks the other three yields by the strength of their correlation with the six-month yield. The signs still matter: a strong negative relationship would point the other way.

These are relationships within our October 2022 data. A different period can give different results!

## There is a lot of fun stuff to explore in seaborn. Enjoy!

Want to look something up? Try the [seaborn tutorial](https://seaborn.pydata.org/tutorial.html), [histogram examples](https://seaborn.pydata.org/generated/seaborn.histplot.html), or [catplot examples](https://seaborn.pydata.org/generated/seaborn.catplot.html). No need to memorize all the arguments. Start with the plot you want, then look up what you need!

Optional post-credit scene: if you're online, uncomment `sns.dogplot()` below. It downloads a random dog image; it isn't part of the lesson and isn't needed to run any of our examples.

In [ ]:
# optional: needs an internet connection
# sns.dogplot()